# 🎓 Automated Candidate-Job Matching & Ranking System
## Applied Natural Language Processing for Resume Screening Using Deep Transformer Models

---
* **Project:** Undergraduate Capstone Project / Thesis Research
* **Architecture:** Cross-Encoder Transformer with Smart Dual-Document Truncation & Class-Weighted Cross-Entropy Loss
* **Recommended Hardware:** Google Colab T4 GPU (**Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU**)
* **Dataset:** Hugging Face `cnamuangtoun/resume-job-description-fit` (8,000 real-world resume-job description pairs)
---


## 0. Environment Setup & Hardware Verification
Install dependencies, suppress tokenizers parallelism warnings, configure random seed, and check CUDA GPU.


In [ ]:
# Install required libraries
!pip install -q transformers datasets accelerate scikit-learn pandas numpy matplotlib seaborn openpyxl

import os
import random
import hashlib
import time
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress Hugging Face tokenizers parallelism warning
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Set deterministic random seeds
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# Hardware diagnostic
print("=" * 65)
print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:       {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: CUDA is not active. Please click Runtime -> Change runtime type -> T4 GPU.")
print("=" * 65)


## 1. Dataset Acquisition & Exploratory Data Analysis (EDA)
Download `cnamuangtoun/resume-job-description-fit` from Hugging Face Datasets.


In [ ]:
from datasets import load_dataset

print("Downloading dataset from Hugging Face Hub...")
raw_dataset = load_dataset("cnamuangtoun/resume-job-description-fit")
print("Dataset successfully downloaded.")
print(f"Raw Train samples: {len(raw_dataset['train'])}")
print(f"Raw Test samples:  {len(raw_dataset['test'])}")

# Global label definitions
LABEL2ID = {"No Fit": 0, "Potential Fit": 1, "Good Fit": 2}
ID2LABEL = {0: "No Fit", 1: "Potential Fit", 2: "Good Fit"}

# Print split distribution
summary_list = []
for split_name in ['train', 'test']:
    counts = Counter(raw_dataset[split_name]['label'])
    total = len(raw_dataset[split_name])
    for label_str, cnt in counts.items():
        summary_list.append({
            "Split": split_name.capitalize(),
            "Label": label_str,
            "Count": cnt,
            "Percentage (%)": round(cnt / total * 100, 2)
        })

df_summary = pd.DataFrame(summary_list)
print("\n" + "=" * 50)
print("CLASS DISTRIBUTION SUMMARY")
print("=" * 50)
print(df_summary.to_string(index=False))


## 2. Data Preprocessing & Quality Engineering
Deduplication via MD5 pair hash, filtering empty/short texts, and removing conflicting annotations.


In [ ]:
def clean_cv_jd_dataset(dataset_split, name="Dataset"):
    df = pd.DataFrame(dataset_split)
    initial_len = len(df)
    
    # 1. MD5 Hash for deduplication
    df['pair_hash'] = df.apply(
        lambda r: hashlib.md5((str(r['resume_text']).strip() + "###" + str(r['job_description_text']).strip()).encode('utf-8')).hexdigest(),
        axis=1
    )
    df = df.drop_duplicates(subset='pair_hash', keep='first')
    dedup_dropped = initial_len - len(df)
    
    # 2. Length filtering
    len_before = len(df)
    df = df[df['resume_text'].astype(str).str.len() > 50]
    df = df[df['job_description_text'].astype(str).str.len() > 50]
    len_dropped = len_before - len(df)
    
    # 3. Resolve conflicting labels
    pair_label_counts = df.groupby('pair_hash')['label'].nunique()
    conflicts = pair_label_counts[pair_label_counts > 1].index
    if len(conflicts) > 0:
        df = df[~df['pair_hash'].isin(conflicts)]
        conflict_dropped = len(conflicts)
    else:
        conflict_dropped = 0
        
    df = df.drop(columns=['pair_hash'])
    
    # Map labels to integer IDs
    label_mapping = {**LABEL2ID, **{v: v for v in LABEL2ID.values()}}
    df['label'] = df['label'].map(label_mapping).astype(int)
    
    print(f"[{name}] Cleaned {initial_len} -> {len(df)} samples | "
          f"Removed {dedup_dropped} duplicates, {len_dropped} short texts, {conflict_dropped} conflicts.")
    return df.reset_index(drop=True)

train_df = clean_cv_jd_dataset(raw_dataset['train'], "Train Set")
test_df = clean_cv_jd_dataset(raw_dataset['test'], "Test Set")

# Plot Class Distribution
plt.figure(figsize=(10, 4))
palette = ['#e74c3c', '#f39c12', '#2ecc71']

plt.subplot(1, 2, 1)
sns.countplot(x='label', data=train_df, palette=palette)
plt.title(f"Train Set Distribution (N={len(train_df)})", fontweight='bold')
plt.xticks(ticks=[0, 1, 2], labels=["No Fit", "Potential", "Good Fit"])
plt.xlabel("")

plt.subplot(1, 2, 2)
sns.countplot(x='label', data=test_df, palette=palette)
plt.title(f"Test Set Distribution (N={len(test_df)})", fontweight='bold')
plt.xticks(ticks=[0, 1, 2], labels=["No Fit", "Potential", "Good Fit"])
plt.xlabel("")

plt.tight_layout()
plt.show()


## 3. Baseline Benchmark (TF-IDF + Logistic Regression)
A classic ML baseline to measure improvement from pretrained deep language models.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score
from scipy.sparse import hstack

# Auto-recovery check
if 'train_df' not in globals() or 'test_df' not in globals():
    raw_dataset = load_dataset("cnamuangtoun/resume-job-description-fit")
    train_df = clean_cv_jd_dataset(raw_dataset['train'], "Train Set")
    test_df = clean_cv_jd_dataset(raw_dataset['test'], "Test Set")

print("=" * 60)
print("TRAINING BASELINE 1: TF-IDF + LOGISTIC REGRESSION")
print("=" * 60)

tfidf_resume = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))
tfidf_jd = TfidfVectorizer(max_features=3000, stop_words='english', ngram_range=(1, 2))

X_train_res = tfidf_resume.fit_transform(train_df['resume_text'])
X_train_jd = tfidf_jd.fit_transform(train_df['job_description_text'])
X_train_base = hstack([X_train_res, X_train_jd])
y_train_base = train_df['label'].values

X_test_res = tfidf_resume.transform(test_df['resume_text'])
X_test_jd = tfidf_jd.transform(test_df['job_description_text'])
X_test_base = hstack([X_test_res, X_test_jd])
y_test_base = test_df['label'].values

lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced', C=1.0)
lr_model.fit(X_train_base, y_train_base)

y_pred_base = lr_model.predict(X_test_base)
baseline_macro_f1 = f1_score(y_test_base, y_pred_base, average='macro')
baseline_acc = accuracy_score(y_test_base, y_pred_base)

print(f"\nBaseline 1 Performance:")
print(f"Accuracy:  {baseline_acc * 100:.2f}%")
print(f"Macro F1:  {baseline_macro_f1:.4f}")
print("\nDetailed Classification Report:")
print(classification_report(y_test_base, y_pred_base, target_names=["No Fit", "Potential Fit", "Good Fit"], digits=4))


## 4. Deep Learning Model Architecture & Fine-Tuning Setup
### Key Innovations:
1. **Pretrained Backbone:** `distilbert-base-uncased` (fast, lightweight transformer).
2. **Smart Dual-Document Truncation:** Resume budget $\le 384$ tokens, JD budget $\le 126$ tokens (Total $\le 512$).
3. **Class-Weighted CrossEntropy Loss:** Alleviates imbalance on minority `Potential Fit` class.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset as TorchDataset
from sklearn.model_selection import train_test_split

# Auto-recovery safeguard
if 'train_df' not in globals() or 'test_df' not in globals():
    print("Auto-initializing train_df and test_df...")
    raw_dataset = load_dataset("cnamuangtoun/resume-job-description-fit")
    train_df = clean_cv_jd_dataset(raw_dataset['train'], "Train Set")
    test_df = clean_cv_jd_dataset(raw_dataset['test'], "Test Set")

MODEL_CHECKPOINT = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print(f"Loaded tokenizer from checkpoint: {MODEL_CHECKPOINT}")

# Compute Class Weights for Imbalanced Loss
label_counts = train_df['label'].value_counts().sort_index().values
total_samples = len(train_df)
num_classes = 3
class_weights = torch.tensor([
    total_samples / (num_classes * label_counts[c]) for c in range(num_classes)
], dtype=torch.float32)

print("\nCalculated Class Weights (Inverse Frequency):")
for c, name in ID2LABEL.items():
    print(f"  Class {c} ({name:13s}): Count = {label_counts[c]:4d} | Weight = {class_weights[c]:.4f}")

# Custom Dataset with Smart Dual-Document Truncation
class SmartCrossEncoderDataset(TorchDataset):
    def __init__(self, dataframe, tokenizer, max_cv=384, max_jd=125, max_total=512):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_cv = max_cv
        self.max_jd = max_jd
        self.max_total = max_total  # Strictly 512 for DistilBERT compatibility!

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        cv_text = str(row['resume_text'])
        jd_text = str(row['job_description_text'])

        cv_tokens = self.tokenizer(cv_text, truncation=True, max_length=self.max_cv, add_special_tokens=False)['input_ids']
        jd_tokens = self.tokenizer(jd_text, truncation=True, max_length=self.max_jd, add_special_tokens=False)['input_ids']

        cls_id = self.tokenizer.cls_token_id or 101
        sep_id = self.tokenizer.sep_token_id or 102
        input_ids = [cls_id] + cv_tokens + [sep_id] + jd_tokens + [sep_id]

        pad_id = self.tokenizer.pad_token_id or 0
        attention_mask = [1] * len(input_ids)
        pad_len = self.max_total - len(input_ids)
        if pad_len > 0:
            input_ids += [pad_id] * pad_len
            attention_mask += [0] * pad_len

        input_ids = input_ids[:self.max_total]
        attention_mask = attention_mask[:self.max_total]

        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'labels': torch.tensor(int(row['label']), dtype=torch.long)
        }

# Stratified split: Train (85%) / Validation (15%)
train_split_df, val_split_df = train_test_split(
    train_df, test_size=0.15, random_state=42, stratify=train_df['label']
)

train_dataset = SmartCrossEncoderDataset(train_split_df, tokenizer)
val_dataset = SmartCrossEncoderDataset(val_split_df, tokenizer)
test_dataset = SmartCrossEncoderDataset(test_df, tokenizer)

print(f"\nSplits prepared: Train = {len(train_dataset)}, Validation = {len(val_dataset)}, Test = {len(test_dataset)}")


## 5. Model Training Loop (Class-Weighted CrossEntropy)
Fine-tune DistilBERT with custom weighted loss, cosine warmup, and checkpoint saving.


In [ ]:
class ClassWeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        weights = class_weights.to(logits.device)
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, 3), labels.view(-1))
        
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, preds, average='macro')
    weighted_f1 = f1_score(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    per_class = f1_score(labels, preds, average=None)
    
    return {
        'accuracy': acc,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
        'f1_no_fit': per_class[0],
        'f1_potential': per_class[1],
        'f1_good_fit': per_class[2]
    }

# Instantiate classification model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

# Training Hyperparameters (version-agnostic)
import inspect
sig_params = inspect.signature(TrainingArguments.__init__).parameters
eval_strategy_key = "eval_strategy" if "eval_strategy" in sig_params else "evaluation_strategy"

args_dict = {
    "output_dir": "./cvjd_model_checkpoints",
    "learning_rate": 2e-5,
    "per_device_train_batch_size": 8,
    "per_device_eval_batch_size": 16,
    "num_train_epochs": 5,
    "weight_decay": 0.01,
    "warmup_steps": 100,
    "gradient_accumulation_steps": 2,
    eval_strategy_key: "epoch",
    "save_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "macro_f1",
    "greater_is_better": True,
    "save_total_limit": 2,
    "logging_steps": 50,
    "seed": 42,
    "fp16": torch.cuda.is_available(),
    "report_to": "none"
}

training_args = TrainingArguments(**args_dict)

trainer = ClassWeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("Starting Transformer Fine-Tuning...")
train_result = trainer.train()

SAVE_PATH = "./best_cvjd_model"
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"\nModel successfully trained and saved to: {SAVE_PATH}")


## 6. Comprehensive Test Evaluation & Benchmark Comparison
Evaluate on the unseen test set (1,759 pairs) and plot confusion matrices.


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print("Running inference on unseen Test Set...")
test_predictions = trainer.predict(test_dataset)
logits = test_predictions.predictions
y_pred_dl = np.argmax(logits, axis=-1)
y_test_true = test_df['label'].values

dl_acc = accuracy_score(y_test_true, y_pred_dl)
dl_macro_f1 = f1_score(y_test_true, y_pred_dl, average='macro')
dl_weighted_f1 = f1_score(y_test_true, y_pred_dl, average='weighted')
per_class_dl = f1_score(y_test_true, y_pred_dl, average=None)

comparison_rows = [
    {
        "Model / Method": "Baseline 1: TF-IDF + Logistic Regression",
        "Accuracy (%)": round(baseline_acc * 100, 2),
        "Macro F1": round(baseline_macro_f1, 4),
        "F1 (No Fit)": round(f1_score(y_test_base, y_pred_base, average=None)[0], 4),
        "F1 (Potential)": round(f1_score(y_test_base, y_pred_base, average=None)[1], 4),
        "F1 (Good Fit)": round(f1_score(y_test_base, y_pred_base, average=None)[2], 4)
    },
    {
        "Model / Method": "Reference Benchmark (domynom/resume-job-distilbert)",
        "Accuracy (%)": 58.40,
        "Macro F1": 0.4320,
        "F1 (No Fit)": 0.6900,
        "F1 (Potential)": 0.2800,
        "F1 (Good Fit)": 0.3260
    },
    {
        "Model / Method": f"Proposed: {MODEL_CHECKPOINT} (Smart Truncation + Weighted Loss)",
        "Accuracy (%)": round(dl_acc * 100, 2),
        "Macro F1": round(dl_macro_f1, 4),
        "F1 (No Fit)": round(per_class_dl[0], 4),
        "F1 (Potential)": round(per_class_dl[1], 4),
        "F1 (Good Fit)": round(per_class_dl[2], 4)
    }
]

df_comp = pd.DataFrame(comparison_rows)
print("\n" + "=" * 80)
print("EMPIRICAL COMPARATIVE PERFORMANCE BENCHMARK")
print("=" * 80)
print(df_comp.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm_base = confusion_matrix(y_test_true, y_pred_base)
cm_dl = confusion_matrix(y_test_true, y_pred_dl)
labels = ["No Fit", "Potential", "Good Fit"]

ConfusionMatrixDisplay(cm_base, display_labels=labels).plot(ax=axes[0], cmap='Oranges', values_format='d')
axes[0].set_title(f"Baseline TF-IDF + LR (Macro F1 = {baseline_macro_f1:.4f})", fontweight='bold')

ConfusionMatrixDisplay(cm_dl, display_labels=labels).plot(ax=axes[1], cmap='Blues', values_format='d')
axes[1].set_title(f"Proposed Model (Macro F1 = {dl_macro_f1:.4f})", fontweight='bold')

plt.suptitle("Confusion Matrix Comparison on Test Set (N=1,759)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 7. Candidate Ranking Evaluation (NDCG@K)
Evaluate ranking quality using Fit Score $S = 100 	imes [P(	ext{Good Fit}) + 0.5 	imes P(	ext{Potential Fit})]$.


In [ ]:
from sklearn.metrics import ndcg_score

probs = F.softmax(torch.tensor(logits), dim=-1).numpy()
scores_s = 100 * (probs[:, 2] + 0.5 * probs[:, 1])

test_df_eval = test_df.copy()
test_df_eval['score_s'] = scores_s
test_df_eval['true_relevance'] = test_df_eval['label']

np.random.seed(42)
test_df_eval['simulated_job_id'] = np.random.randint(0, len(test_df_eval) // 10, size=len(test_df_eval))

ndcg_5_scores = []
ndcg_10_scores = []

for job_id, group in test_df_eval.groupby('simulated_job_id'):
    if len(group) >= 5 and group['true_relevance'].nunique() > 1:
        y_true = np.asarray([group['true_relevance'].values])
        y_score = np.asarray([group['score_s'].values])
        ndcg_5_scores.append(ndcg_score(y_true, y_score, k=5))
        ndcg_10_scores.append(ndcg_score(y_true, y_score, k=min(10, len(group))))

print("=" * 50)
print("RANKING QUALITY METRICS (NDCG)")
print("=" * 50)
print(f"Mean NDCG@5:   {np.mean(ndcg_5_scores):.4f}")
print(f"Mean NDCG@10:  {np.mean(ndcg_10_scores):.4f}")


## 8. Qualitative Demo & Real-World Inference Engine
Predict on new candidate resumes and job descriptions.


In [ ]:
def predict_cv_jd_fit(resume_text, jd_text, eval_model, eval_tokenizer):
    eval_model.eval()
    
    cv_enc = eval_tokenizer(resume_text, truncation=True, max_length=384, add_special_tokens=False)['input_ids']
    jd_enc = eval_tokenizer(jd_text, truncation=True, max_length=125, add_special_tokens=False)['input_ids']
    
    cls_id = eval_tokenizer.cls_token_id or 101
    sep_id = eval_tokenizer.sep_token_id or 102
    input_ids = [cls_id] + cv_enc + [sep_id] + jd_enc + [sep_id]
    input_ids = input_ids[:512]
    
    input_tensor = torch.tensor([input_ids], dtype=torch.long)
    attention_mask = torch.ones_like(input_tensor)
    
    if torch.cuda.is_available():
        input_tensor = input_tensor.cuda()
        attention_mask = attention_mask.cuda()
        eval_model = eval_model.cuda()
        
    with torch.no_grad():
        outputs = eval_model(input_ids=input_tensor, attention_mask=attention_mask)
        probs = F.softmax(outputs.logits, dim=-1).squeeze().cpu().numpy()
        pred_label_id = int(np.argmax(probs))
        
    score_s = round(float(100 * (probs[2] + 0.5 * probs[1])), 1)
    
    return {
        "Predicted Label": ID2LABEL[pred_label_id],
        "Fit Score S (/100)": score_s,
        "P(No Fit)": round(float(probs[0]), 4),
        "P(Potential Fit)": round(float(probs[1]), 4),
        "P(Good Fit)": round(float(probs[2]), 4)
    }

test_cases = [
    {
        "Scenario": "Case 1: Strong Match (Senior Python Backend Dev)",
        "Resume": "Senior Backend Engineer with 5 years experience in Python, FastAPI, Django, PostgreSQL, Docker, Redis. Built high-scale microservices and RESTful APIs.",
        "JD": "Seeking Senior Python Developer with 3+ years experience. Strong background in FastAPI, PostgreSQL, and scalable microservices architecture."
    },
    {
        "Scenario": "Case 2: Complete Mismatch (Civil Engineer -> Python Dev)",
        "Resume": "Site Construction Manager with 7 years experience in structural engineering, AutoCAD, concrete testing, safety site supervision, and vendor contractor management.",
        "JD": "Seeking Senior Python Developer with 3+ years experience. Strong background in FastAPI, PostgreSQL, and scalable microservices architecture."
    },
    {
        "Scenario": "Case 3: Potential / Marginal Fit (Junior Data Analyst -> Backend Dev)",
        "Resume": "Junior Data Analyst with 1 year experience in Python pandas, Excel, and SQL queries. Passionate about transitioning to software engineering.",
        "JD": "Seeking Senior Python Developer with 3+ years experience. Strong background in FastAPI, PostgreSQL, and scalable microservices architecture."
    }
]

print("=" * 80)
print("INFERENCE DEMO ON QUALITATIVE TEST SCENARIOS")
print("=" * 80)
for case in test_cases:
    res = predict_cv_jd_fit(case['Resume'], case['JD'], model, tokenizer)
    print(f"\nScenario:           {case['Scenario']}")
    print(f"Predicted Class:    {res['Predicted Label']}")
    print(f"Calculated Score:   {res['Fit Score S (/100)']} / 100")
    print(f"Probabilities:      No Fit: {res['P(No Fit)']} | Potential: {res['P(Potential Fit)']} | Good: {res['P(Good Fit)']}")


## 9. Stratified 100-Sample Test & Excel Report Generation
Batch evaluate 100 stratified test pairs and export an Excel report with download trigger.


In [ ]:
# Self-contained prediction function with strict 512 token budget
def predict_cv_jd_fit_safe(resume_text, jd_text, eval_model, eval_tokenizer):
    eval_model.eval()
    
    cv_enc = eval_tokenizer(str(resume_text), truncation=True, max_length=384, add_special_tokens=False)['input_ids']
    jd_enc = eval_tokenizer(str(jd_text), truncation=True, max_length=125, add_special_tokens=False)['input_ids']
    
    cls_id = eval_tokenizer.cls_token_id or 101
    sep_id = eval_tokenizer.sep_token_id or 102
    input_ids = [cls_id] + cv_enc + [sep_id] + jd_enc + [sep_id]
    input_ids = input_ids[:512]
    
    input_tensor = torch.tensor([input_ids], dtype=torch.long)
    attention_mask = torch.ones_like(input_tensor)
    
    device = next(eval_model.parameters()).device
    input_tensor = input_tensor.to(device)
    attention_mask = attention_mask.to(device)
        
    with torch.no_grad():
        outputs = eval_model(input_ids=input_tensor, attention_mask=attention_mask)
        probs = F.softmax(outputs.logits, dim=-1).squeeze().cpu().numpy()
        pred_label_id = int(np.argmax(probs))
        
    score_s = round(float(100 * (probs[2] + 0.5 * probs[1])), 1)
    
    return {
        "Predicted Label": ID2LABEL[pred_label_id],
        "Fit Score S (/100)": score_s,
        "P(No Fit)": round(float(probs[0]), 4),
        "P(Potential Fit)": round(float(probs[1]), 4),
        "P(Good Fit)": round(float(probs[2]), 4)
    }

# Sample 100 balanced test samples (34 No Fit, 33 Potential Fit, 33 Good Fit)
df_test_clean = test_df.copy()
df_test_clean['label_name'] = df_test_clean['label'].map(ID2LABEL)

sample_100 = pd.concat([
    df_test_clean[df_test_clean['label_name'] == 'No Fit'].sample(n=34, random_state=42),
    df_test_clean[df_test_clean['label_name'] == 'Potential Fit'].sample(n=33, random_state=42),
    df_test_clean[df_test_clean['label_name'] == 'Good Fit'].sample(n=33, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

sample_100['sample_id'] = [f"TEST_EN_{i+1:03d}" for i in range(len(sample_100))]
print(f"Selected {len(sample_100)} stratified test samples:")
print(sample_100['label_name'].value_counts())

eval_records = []
print("\nRunning batch evaluation on 100 samples...")
for idx, row in sample_100.iterrows():
    pred_res = predict_cv_jd_fit_safe(row['resume_text'], row['job_description_text'], model, tokenizer)
    ground_truth = row['label_name']
    pred_label = pred_res['Predicted Label']
    match_status = "MATCH" if ground_truth == pred_label else "MISMATCH"
    
    eval_records.append({
        "Sample ID": row['sample_id'],
        "Ground Truth Label": ground_truth,
        "AI Predicted Label": pred_label,
        "Evaluation": match_status,
        "Fit Score S (/100)": pred_res['Fit Score S (/100)'],
        "P(No Fit)": pred_res['P(No Fit)'],
        "P(Potential Fit)": pred_res['P(Potential Fit)'],
        "P(Good Fit)": pred_res['P(Good Fit)'],
        "Resume Snippet": str(row['resume_text'])[:150].replace('\n', ' ') + "...",
        "JD Snippet": str(row['job_description_text'])[:150].replace('\n', ' ') + "...",
        "Full Resume Text": str(row['resume_text']),
        "Full JD Text": str(row['job_description_text'])
    })

df_eval_100 = pd.DataFrame(eval_records)
sample_acc = (df_eval_100['Evaluation'] == 'MATCH').mean() * 100
print(f"\nBatch Evaluation Complete: Accuracy = {sample_acc:.2f}% on 100 samples.")

EXCEL_OUTPUT = "test_100_samples_evaluation.xlsx"
df_eval_100.to_excel(EXCEL_OUTPUT, index=False, engine='openpyxl')
print(f"Excel report saved successfully to: {EXCEL_OUTPUT}")

try:
    from google.colab import files
    print("Triggering browser download for Google Colab...")
    files.download(EXCEL_OUTPUT)
except Exception:
    print(f"File available in current working directory: {os.path.abspath(EXCEL_OUTPUT)}")


In [ ]:
# ==============================================================================
# BƯỚC BỔ SUNG: NÉN THƯ MỤC best_cvjd_model, LƯU GOOGLE DRIVE & TẢI VỀ MÁY
# Sau khi tải best_cvjd_model.zip về máy, giải nén vào:
# /Users/khoait/Documents/SourceCompany/khoaluantotnghiep/ai-service/models/best_cvjd_model/
# ==============================================================================
import shutil
import os

ZIP_NAME = "best_cvjd_model"
if os.path.exists("./best_cvjd_model"):
    zip_path = shutil.make_archive(ZIP_NAME, "zip", "./best_cvjd_model")
    print(f"✅ Đã nén model thành công: {zip_path} ({os.path.getsize(zip_path) / (1024*1024):.2f} MB)")

    # 1. Sao lưu lên Google Drive (nếu đang chạy trên Colab)
    try:
        from google.colab import drive, files
        drive.mount("/content/drive", force_remount=False)
        drive_target = "/content/drive/MyDrive/best_cvjd_model.zip"
        shutil.copy2(zip_path, drive_target)
        print(f"✅ Đã sao lưu vĩnh viễn lên Google Drive: {drive_target}")
        files.download(zip_path)
    except Exception as e:
        print(f"ℹ️ Không chạy trên Colab hoặc bỏ qua Google Drive: {e}")
else:
    print("⚠️ Chưa tìm thấy ./best_cvjd_model. Hãy chạy Cell huấn luyện trước.")
